![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 1. Parte B: Modelo, entrenamiento y evaluación

**Asignatura:** Procesamiento del Lenguaje Natural
**Bloque:** 1 — Introducción a las Redes Neuronales

---

## Introducción

En la Parte A convertiste texto en un tensor `(B, L)` de índices. Acá ese tensor entra por fin a un modelo.

Vas a construir un clasificador de escenarios: recibe una orden en español —*"despertame a las nueve"*, *"qué tiempo hace mañana"*— y predice a cuál de los 18 dominios pertenece. La arquitectura es la más simple que puede funcionar sobre texto: una tabla de *embeddings*, un promedio enmascarado que colapsa la secuencia en un vector, y un perceptrón multicapa encima. Nada de recurrencia, nada de atención. Eso llega después.

La razón de empezar por acá no es que sea fácil, sino que es **completa**: tiene todas las piezas de un entrenamiento real —datos servidos por lotes, una pérdida, un optimizador, un conjunto de validación, métricas por clase, sobreajuste y regularización— y ninguna que distraiga. Cuando en la Unidad 2 le agreguemos una capa recurrente, lo único que va a cambiar es la línea que colapsa la secuencia.

Y termina con un límite. El último ejercicio te va a mostrar que este modelo no puede distinguir *"apagá la luz de la cocina"* de *"la cocina apagá de luz la"*: para él son literalmente la misma entrada. Ese techo es el que motiva toda la Unidad 2.

Al completar este laboratorio vas a poder:

- Implementar el protocolo `Dataset` de PyTorch y servir datos con `DataLoader`.
- Construir un clasificador de texto con `nn.Embedding` y promedio enmascarado.
- Verificar que un modelo sin entrenar da la pérdida que la teoría predice.
- Escribir un loop de entrenamiento completo y una función de evaluación.
- Comparar optimizadores y tasas de aprendizaje con evidencia.
- Leer una matriz de confusión de 18 clases y sacar conclusiones por clase.
- Provocar sobreajuste, medirlo, y bajarlo con *weight decay* y *dropout*.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para resolver cada ejercicio, consultá el material teórico de las Clases 3 a 6 de la Unidad 1.
- **Este laboratorio corre entero en CPU.** El entrenamiento completo son menos de treinta segundos; no hace falta GPU.
- La celda de setup te da ya escrita la clase `Vocabulario` de la Parte A. Usala tal cual: si tu implementación difiere, los números de este laboratorio no van a coincidir con los esperados.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas (enunciados, explicaciones, ejemplos provistos y el encabezado) **no se tocan**: la corrección se hace celda por celda de manera automática y modificar lo que no corresponde puede invalidar tu entrega.

Si necesitás probar algo fuera de una celda de actividad, hacelo en una copia aparte y revertí los cambios antes de entregar.

In [ ]:
# ─── Setup: imports ─────────────────────────────────────────────────────────
import re
import math
import copy
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f"Versión de PyTorch: {torch.__version__}")

In [ ]:
# ─── Setup: el corpus y el pipeline de la Parte A ───────────────────────────
# Esta celda reconstruye, tal cual, lo que construiste en la Parte A. Está
# provista para que los números de este laboratorio sean comparables entre
# todos: si cada uno usara su propio freq_min, nada sería contrastable.
REPO = "https://github.com/javovelez/pln-labs/raw/main/datos"
HUB  = "https://huggingface.co/datasets/SetFit/amazon_massive_scenario_es-ES/resolve/main"


def leer_split(nombre):
    """Lee un split del corpus, del repo de la materia o del Hub como respaldo."""
    try:
        return pd.read_json(f"{REPO}/{nombre}.jsonl", lines=True)
    except Exception:
        return pd.read_json(f"{HUB}/{nombre}.jsonl", lines=True)


train = leer_split("train")
val   = leer_split("validation")
test  = leer_split("test")

ESCENARIOS = (train[["label", "label_text"]]
              .drop_duplicates()
              .sort_values("label")["label_text"]
              .tolist())
N_CLASES = len(ESCENARIOS)


def tok_simple(texto):
    """Minúsculas y secuencias alfanuméricas. El tokenizador de la Parte A."""
    return re.findall(r"\w+", texto.lower())


class Vocabulario:
    """Mapea tokens a índices y viceversa. Idéntica a la de la Parte A."""

    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, textos, tokenizador=tok_simple, freq_min=1, max_tokens=None):
        self.tokenizador = tokenizador
        self.freq_min = freq_min
        self.contador = collections.Counter(
            t for texto in textos for t in tokenizador(texto)
        )
        candidatos = [(p, f) for p, f in self.contador.most_common() if f >= freq_min]
        if max_tokens is not None:
            candidatos = candidatos[:max_tokens]
        self.itos = [self.PAD, self.UNK] + [p for p, _ in candidatos]
        self.stoi = {p: i for i, p in enumerate(self.itos)}
        self.pad_id = self.stoi[self.PAD]
        self.unk_id = self.stoi[self.UNK]

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token):
        return self.stoi.get(token, self.unk_id)

    def __repr__(self):
        return (f"Vocabulario({len(self):,} tokens, "
                f"tokenizador={self.tokenizador.__name__}, freq_min={self.freq_min})")

    def codificar(self, texto):
        return [self[t] for t in self.tokenizador(texto)]

    def decodificar(self, indices, ocultar_pad=True):
        if torch.is_tensor(indices):
            indices = indices.tolist()
        return [self.itos[i] for i in indices
                if not (ocultar_pad and i == self.pad_id)]

    def codificar_lote(self, textos, largo):
        salida = torch.zeros(len(textos), largo, dtype=torch.long)
        for i, texto in enumerate(textos):
            ids = self.codificar(texto)[:largo]
            salida[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        return salida


# ─── Los tensores, con las decisiones de la Parte A ─────────────────────────
L = 16                                   # el 2,2% de las órdenes se trunca
vocab = Vocabulario(train.text, tokenizador=tok_simple, freq_min=2)

X_ent  = vocab.codificar_lote(train.text, L)
X_val  = vocab.codificar_lote(val.text,   L)
X_test = vocab.codificar_lote(test.text,  L)

y_ent  = torch.tensor(train.label.values)
y_val  = torch.tensor(val.label.values)
y_test = torch.tensor(test.label.values)

print(vocab)
print()
print(f"entrenamiento: X {tuple(X_ent.shape)}  y {tuple(y_ent.shape)}")
print(f"validación:    X {tuple(X_val.shape)}  y {tuple(y_val.shape)}")
print(f"prueba:        X {tuple(X_test.shape)}  y {tuple(y_test.shape)}")
print()
print(f"{N_CLASES} escenarios: {', '.join(ESCENARIOS)}")

# La clase más frecuente es la referencia mínima: un modelo que siempre
# predijera "calendar" acertaría esto. Cualquier cosa por debajo es un fracaso.
mayoritaria = y_val.bincount().argmax().item()
print()
print(f"clase mayoritaria en validación: {ESCENARIOS[mayoritaria]} "
      f"({y_val.bincount().max().item() / len(y_val):.1%})")

---
## Sección A: Los datos y el modelo

Tres ejercicios para tener el modelo en pie: servir los datos por lotes, definir la arquitectura, y verificar que la arquitectura hace lo que creemos antes de entrenarla.

El tercero parece un trámite y es de los más formativos del laboratorio: es la verificación que distingue a alguien que entiende la entropía cruzada de alguien que la invoca.

### Ejercicio 1 — El `Dataset` y los tres `DataLoader`

**Objetivo:** Implementar el protocolo `Dataset` de PyTorch y armar los tres cargadores de datos, prestando atención al orden en que el corpus viene dado.

**Enunciado:**

**Parte A — el `Dataset`.** Implementá `OrdenesDataset(Dataset)`, que sirve pares `(tensor de índices, etiqueta)` a partir de dos tensores ya codificados:

1. `__init__(self, X, y)` guarda los dos tensores. Verificá con un `assert` que tienen el mismo largo.
2. `__len__` devuelve la cantidad de ejemplos.
3. `__getitem__(self, i)` devuelve la tupla `(X[i], y[i])`.

Instanciá los tres (`ds_ent`, `ds_val`, `ds_test`) e imprimí, para el ejemplo 0 del de entrenamiento: la forma del tensor, la etiqueta, el texto decodificado y el nombre del escenario.

**Parte B — los `DataLoader`.** Antes de armarlos, mirá un detalle del corpus:

1. Imprimí las primeras 30 etiquetas de `y_ent`. Vas a ver que **el corpus viene agrupado por escenario**.
2. Armá `dl_ent`, `dl_val` y `dl_test` con `batch_size=64`. Decidí el valor de `shuffle` para cada uno y dejá el criterio escrito como comentario. Fijá `torch.manual_seed(0)` antes para que el barajado sea reproducible.
3. **Mostrá por qué importa:** armá además un `DataLoader` sobre `ds_ent` con `shuffle=False` e imprimí cuántas clases distintas trae su primer lote, comparado con cuántas trae el primer lote de `dl_ent`.
4. Imprimí cuántos lotes tiene cada uno de los tres cargadores.

> **Pista:** El protocolo `Dataset` de PyTorch es exactamente eso: una clase con `__len__` y `__getitem__`. No hay nada más. El `DataLoader` solo necesita esas dos operaciones para armar lotes, barajar e iterar.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Supongamos que entrenás con `shuffle=False` sobre este corpus agrupado por escenario.

a) Describí qué le pasa al gradiente a lo largo de una época y qué forma tendría la curva de pérdida por lote.

b) ¿Por qué en validación y prueba **no** hace falta barajar? ¿Cambiaría en algo la métrica final si barajaras?

*(Escribí tu respuesta acá)*

### Ejercicio 2 — El clasificador: `nn.Embedding`, promedio enmascarado y MLP

**Objetivo:** Construir el modelo completo y ver dónde están sus parámetros.

**Enunciado:**

Implementá `ClasificadorOrdenes(nn.Module)` con esta firma:

```python
def __init__(self, n_vocab, dim_emb=64, dim_oculta=128, n_clases=18, pad_id=0)
```

y estas tres capas:

- `self.embedding`: una `nn.Embedding` de `n_vocab × dim_emb`. Pasale `padding_idx=pad_id` — esto fuerza la fila del `<pad>` a ser todo ceros y, además, hace que nunca reciba gradiente.
- `self.oculta`: una `nn.Linear` de `dim_emb` a `dim_oculta`.
- `self.salida`: una `nn.Linear` de `dim_oculta` a `n_clases`.

Y estos dos métodos:

- `promediar(self, x)`: recibe `(B, L)` y devuelve `(B, E)`. Es exactamente el promedio enmascarado del Ejercicio 3 de la Parte A: pasar los índices por la tabla, construir la máscara `(B, L, 1)`, multiplicar, sumar sobre la dimensión de la secuencia y dividir por la cantidad de tokens reales con `.clamp(min=1)`.
- `forward(self, x)`: promedia, aplica `self.oculta` seguida de `F.relu`, y devuelve `self.salida(...)`. **Sin softmax** — el modelo devuelve *logits*.

Después:

1. Instanciá el modelo con `torch.manual_seed(0)` e imprimilo.
2. Imprimí una tabla con un renglón por parámetro (`modelo.named_parameters()`): nombre, forma y cantidad de elementos. Al final, el total y qué porcentaje de ese total está en la tabla de *embeddings*.
3. Pasá un lote de `dl_ent` por el modelo y verificá que la forma de salida es `(B, n_clases)`.

> **Pista 1:** `padding_idx` no es cosmético. Sin él, la fila del `<pad>` recibiría gradiente en cada lote y se movería, con lo cual el relleno pasaría a aportar un vector no nulo al promedio.

> **Pista 2:** Que el modelo devuelva *logits* y no probabilidades es deliberado: `nn.CrossEntropyLoss` aplica el softmax internamente, de manera numéricamente estable. Aplicarlo dos veces es un error clásico.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

El 95% de los parámetros del modelo está en la tabla de *embeddings*, y solo el 5% en las dos capas lineales, que son la parte que uno llamaría "la red".

a) ¿Por qué la tabla es tan grande, y de qué dos cantidades depende su tamaño?

b) Cada fila de la tabla se actualiza solo cuando aparece su palabra en un lote. ¿Qué consecuencia tiene eso para las palabras poco frecuentes, y qué relación tiene con la decisión de `freq_min` que tomaste en la Parte A?

*(Escribí tu respuesta acá)*

### Ejercicio 3 — Verificación de cordura: la pérdida antes de entrenar

**Objetivo:** Predecir analíticamente cuánto tiene que dar la pérdida de un modelo recién inicializado, y comprobarlo.

**Enunciado:**

Un modelo recién inicializado no sabe nada: sus pesos son aleatorios y sus *logits* van a ser aproximadamente iguales entre sí, así que el softmax va a repartir la probabilidad casi uniformemente entre las 18 clases.

1. **Deducí el valor esperado.** Escribí como comentario la fórmula de la entropía cruzada para un solo ejemplo y calculá cuánto vale si el modelo le asigna probabilidad $1/C$ a cada clase, con $C = 18$.
2. **Medilo.** Instanciá `nn.CrossEntropyLoss()` en `criterio`, tomá un lote de `dl_ent` y calculá la pérdida del modelo sin entrenar, con `torch.no_grad()`.
3. **Compará** el valor medido con el predicho e imprimí la diferencia.
4. **Mostrá de dónde sale.** Aplicá `F.softmax` a los *logits* del primer ejemplo del lote e imprimí las 18 probabilidades redondeadas, junto a su mínimo, su máximo y su suma.

> **Pista:** La entropía cruzada para un ejemplo es $-\log p_{\text{correcta}}$, donde $p_{\text{correcta}}$ es la probabilidad que el modelo le asignó a la clase verdadera. Con probabilidades uniformes esa probabilidad no depende de cuál sea la clase correcta.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Esta verificación parece un trámite y es una de las herramientas de diagnóstico más útiles que hay.

a) Si al correrla te hubiera dado una pérdida inicial de 0,4, ¿qué habría que sospechar? ¿Y si te hubiera dado 15?

b) La clase mayoritaria del corpus (`calendar`) es el 13,8% de los datos, no el 5,6% que corresponde a 18 clases balanceadas. ¿Por qué la pérdida inicial da igual `ln(18)` y no algo relacionado con esa proporción?

*(Escribí tu respuesta acá)*

---
## Sección B: Entrenamiento

Cuatro ejercicios: el paso de descenso hecho a mano para ver el mecanismo, el loop completo, y la comparación de tasas de aprendizaje y optimizadores.

### Ejercicio 4 — Un paso de descenso de gradiente, sin optimizador

**Objetivo:** Ejecutar a mano el paso que después va a hacer el optimizador, para ver que no hay ninguna magia adentro.

**Enunciado:**

1. **Copiá el modelo** con `copy.deepcopy(modelo)` en una variable `m1`, para no arruinar el original.
2. **Tomá un lote** de `dl_ent` y calculá la pérdida. Imprimila.
3. **Poné los gradientes en cero** con `m1.zero_grad()` y llamá a `.backward()` sobre la pérdida.
4. **Mirá un gradiente concreto**: imprimí `m1.salida.bias.grad` redondeado, que tiene un valor por clase.
5. **Aplicá el paso a mano**, con `lr = 0.5`: adentro de un bloque `torch.no_grad()`, recorré `m1.parameters()` y a cada uno restale `lr` por su gradiente.
6. **Recalculá la pérdida sobre el mismo lote** e imprimila. Tiene que haber bajado.
7. **Repetí el paso 30 veces sobre ese mismo lote**, guardando la pérdida en una lista, y graficala. Marcá `ln(18)` con una línea horizontal punteada.

> **Pista 1:** El bloque `torch.no_grad()` es imprescindible en el punto 5: la actualización de los pesos es una operación sobre tensores, y sin él `autograd` la registraría en el grafo como si fuera parte del cálculo.

> **Pista 2:** Que la pérdida sobre **un solo lote** baje hasta casi cero no es una buena noticia sobre el modelo: es memorización de esos 64 ejemplos. Acá el objetivo es solo ver el mecanismo.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

¿Qué pasaría si sacaras el bloque `torch.no_grad()` de la actualización? Explicá qué construiría `autograd` en ese caso y por qué el resultado sería incorrecto además de caro.

*(Escribí tu respuesta acá)*

### Ejercicio 5 — `evaluar()` y `entrenar()`

**Objetivo:** Escribir el loop de entrenamiento completo y la función de evaluación que vas a reusar en todo el resto del laboratorio.

**Enunciado:**

**Parte A — `evaluar(modelo, dataloader)`.** Devuelve la tupla `(pérdida media, accuracy)` sobre el cargador completo:

1. Poné el modelo en modo evaluación con `.eval()`.
2. Recorré el cargador adentro de `torch.no_grad()`.
3. Acumulá la pérdida **ponderada por el tamaño del lote** (el último lote puede ser más chico) y la cantidad de aciertos.
4. Devolvé los dos promedios.

Probala sobre `dl_val` con el modelo sin entrenar: tiene que dar una pérdida cercana a `ln(18)` y una accuracy cercana al azar.

**Parte B — `entrenar(...)`.** Con esta firma:

```python
def entrenar(modelo, dl_ent, dl_val, epocas=8, lr=1e-3, optimizador="adam",
             weight_decay=0.0, verbose=True)
```

1. Construí el optimizador según el string recibido (`"adam"` → `torch.optim.Adam`, `"sgd"` → `torch.optim.SGD`), pasándole `lr` y `weight_decay`.
2. Por cada época: modelo en modo `.train()`, y por cada lote las tres líneas de siempre —`zero_grad()`, `backward()`, `step()`—, guardando la pérdida de cada lote.
3. Al final de cada época, evaluá sobre entrenamiento y sobre validación, y guardá las cuatro métricas.
4. Devolvé un diccionario `hist` con las claves `"ent"`, `"val"`, `"acc_ent"`, `"acc_val"` y `"por_lote"`.
5. Si `verbose`, imprimí una línea por época con las cuatro métricas.

Entrená el modelo desde cero (`torch.manual_seed(0)` y una instancia nueva) por 8 épocas con Adam y `lr=1e-3`. Después:

6. Graficá, lado a lado, la pérdida por lote (con `ln(18)` marcado) y las dos curvas de pérdida por época.
7. Evaluá sobre `dl_test` e imprimí la pérdida y la accuracy finales, comparándolas contra la referencia de la clase mayoritaria.

> **Pista 1:** `.train()` y `.eval()` cambian el comportamiento de capas como `Dropout`. Ahora no hay ninguna, así que no cambia nada — pero en el Ejercicio 8 sí la va a haber, y si la función no está bien escrita desde ahora, el bug aparece allá y es difícil de rastrear.

> **Pista 2:** Para ponderar la pérdida por el tamaño del lote: acumulá `criterio(logits, yb).item() * len(yb)` y dividí al final por el total de ejemplos.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Mirá los dos gráficos y comparalos entre sí.

a) La curva por lote es mucho más ruidosa que la curva por época, aunque las dos midan la misma cantidad. Explicá por qué, y qué información da cada una que la otra no da.

b) Al final del entrenamiento la pérdida de validación es visiblemente mayor que la de entrenamiento, y la brecha se agranda con las épocas. ¿Qué está pasando, y por qué la accuracy de validación puede seguir subiendo mientras eso ocurre?

*(Escribí tu respuesta acá)*

### Ejercicio 6 — Tasa de aprendizaje y optimizador

**Objetivo:** Comparar con evidencia el efecto de la tasa de aprendizaje y del optimizador, en lugar de aceptar los valores por defecto.

**Enunciado:**

1. **Entrená cuatro configuraciones** por 5 épocas cada una, siempre desde un modelo nuevo con `torch.manual_seed(0)` para que la comparación sea justa, y con `verbose=False`:

   | Configuración | optimizador | `lr` |
   |---|---|---|
   | SGD lento | `"sgd"` | 0,1 |
   | SGD razonable | `"sgd"` | 1,0 |
   | Adam razonable | `"adam"` | 1e-3 |
   | Adam demasiado alto | `"adam"` | 1e-1 |

2. **Guardá el historial de cada una** en un diccionario e imprimí una tabla con la pérdida y la accuracy de validación finales de las cuatro.

3. **Graficá lado a lado** las cuatro curvas de pérdida de validación y las cuatro de accuracy de validación, por época.

> **Pista 1:** Cinco épocas de cada configuración son unos pocos segundos en CPU. No hace falta reducir el corpus.

> **Pista 2:** Ojo al comparar los `lr` entre optimizadores: la tasa "razonable" de SGD y la de Adam difieren en tres órdenes de magnitud, y eso no significa que uno sea más rápido que el otro. Adam divide el paso por la magnitud del gradiente, así que su `lr` significa otra cosa.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Describí el comportamiento de cada una de las cuatro configuraciones y clasificalas en los tres regímenes característicos de la tasa de aprendizaje: demasiado baja, razonable, demasiado alta.

b) `SGD lr=0.1` y `Adam lr=1e-3` difieren en un factor de 100 en la tasa, y sin embargo Adam avanza mucho más rápido. ¿Por qué comparar los `lr` entre optimizadores distintos no tiene sentido?

*(Escribí tu respuesta acá)*

---
## Sección C: Evaluación, sobreajuste y el límite del modelo

Ya tenés un modelo que anda. Los últimos tres ejercicios son sobre entenderlo: qué se equivoca y por qué, cuánto está memorizando, y qué es lo que no va a poder aprender nunca con esta arquitectura.

### Ejercicio 7 — Matriz de confusión y análisis por clase

**Objetivo:** Ir más allá de la accuracy global y entender qué escenarios se confunden entre sí, y por qué.

**Enunciado:**

Volvé al `modelo` entrenado del Ejercicio 5.

1. **Escribí `predecir_todo(modelo, dataloader)`**, que devuelve dos tensores: las etiquetas reales y las predichas, sobre el cargador completo.
2. **Escribí `matriz_confusion(reales, predichas, n_clases)`**, que devuelve un tensor `(n_clases, n_clases)` donde la posición `[i, j]` cuenta cuántos ejemplos de la clase real `i` se predijeron como `j`.
3. **Graficá la matriz** sobre el conjunto de prueba con `imshow`, con los nombres de los escenarios en los dos ejes. Con 18 clases conviene `figsize=(9, 8)` y rotar las etiquetas del eje x.
4. **Escribí `precision_recall(cm)`** y mostrá una tabla con precisión, *recall*, F1 y cantidad de ejemplos por escenario, **ordenada por F1 de peor a mejor**.
5. **Listá las seis confusiones más grandes** fuera de la diagonal, en la forma `real -> predicho: cantidad`.

> **Pista 1:** Para una clase $k$: los verdaderos positivos son `cm[k, k]`; los falsos positivos, la suma de la columna $k$ menos la diagonal; los falsos negativos, la suma de la fila $k$ menos la diagonal. La precisión es $TP/(TP+FP)$, el *recall* es $TP/(TP+FN)$, y F1 es su media armónica.

> **Pista 2:** Cuidado con dividir por cero si alguna clase nunca se predice.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Mirá el escenario con peor F1 y las confusiones más grandes. ¿El problema es del modelo o del esquema de clases? Justificá mirando qué órdenes caerían en cada uno de los escenarios involucrados.

b) Elegí una clase con **precisión alta y recall bajo** y explicá qué significa esa combinación en términos concretos. Después, mirando la columna de ejemplos: ¿por qué la accuracy global del 80% esconde a las clases con F1 bajo?

*(Escribí tu respuesta acá)*

### Ejercicio 8 — Provocar el sobreajuste, medirlo y bajarlo

**Objetivo:** Ver el sobreajuste como un fenómeno que se produce a voluntad y se mide, no como algo que "pasa", y comprobar el efecto de dos regularizadores.

**Enunciado:**

**Parte A — provocarlo.** Entrená un modelo deliberadamente sobredimensionado durante muchas épocas:

1. Instanciá `ClasificadorOrdenes` con `dim_emb=128` y `dim_oculta=256`, con `torch.manual_seed(0)`.
2. Imprimí su cantidad total de parámetros y compará contra la cantidad de ejemplos de entrenamiento.
3. Entrenalo 25 épocas con Adam y `lr=1e-3`, sin regularización, con `verbose=False`.
4. Imprimí una tabla con las métricas de las épocas 1, 5, 10, 15, 20 y 25.
5. Reportá la **brecha final de accuracy** (entrenamiento menos validación) y en qué época se alcanzó la mejor accuracy de validación.

**Parte B — bajarlo.** Ahora las dos herramientas:

1. Definí `ClasificadorRegularizado`, idéntico al anterior pero con un `nn.Dropout(p_dropout)` aplicado sobre el vector promediado, antes de la capa oculta. Usá `p_dropout=0.5`.
2. Entrenalo con la misma configuración pero agregando `weight_decay=1e-4`.
3. Imprimí la misma tabla de épocas y la brecha final.
4. **Graficá las dos corridas juntas**: en un panel las cuatro curvas de pérdida (las dos de cada modelo), en el otro las cuatro de accuracy. Usá línea llena para validación y punteada para entrenamiento.
5. Imprimí una tabla comparativa final con, para cada modelo: accuracy de entrenamiento, accuracy de validación y brecha.

> **Pista 1:** El `dropout` va **después** del promedio y **antes** de la capa oculta. Y solo actúa en modo `.train()`: por eso importaba que `evaluar()` llame a `.eval()`.

> **Pista 2:** `weight_decay` es el nombre que PyTorch le da a la regularización L2, y se pasa directamente al constructor del optimizador.

> **Pista 3:** Prestá atención a las accuracies de **entrenamiento** al comparar. La regularización casi siempre las baja: esa es exactamente su función.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Compará las dos tablas mirando las columnas de **entrenamiento** y de **validación** por separado. ¿Qué le hizo la regularización a cada una, y por qué eso es lo que se busca?

b) En la corrida sin regularizar, la pérdida de validación toca un mínimo y después sube, mientras que la accuracy de validación se queda más o menos igual. Si tuvieras que quedarte con un modelo, ¿en qué época pararías y con qué criterio?

*(Escribí tu respuesta acá)*

In [ ]:
# ─── Provisto: el mismo conjunto de palabras, distinto orden ────────────────
# Esta celda no hay que completarla, pero usa el `modelo` que entrenaste en el
# Ejercicio 5: si todavía no lo hiciste, va a dar NameError. Corrala cuando
# llegues acá, y mirá el resultado antes de responder el Ejercicio 9.
pares = [
    ("apaga la luz de la cocina",     "la cocina apaga de luz la"),
    ("recuérdame llamar a mamá",      "mamá llamar recuérdame a"),
    ("pon música y sube el volumen",  "sube el volumen y pon música"),
]

modelo.eval()
for a, b in pares:
    xa = vocab.codificar_lote([a], L)
    xb_ = vocab.codificar_lote([b], L)
    with torch.no_grad():
        la, lb = modelo(xa), modelo(xb_)

    print(f"{a!r:34s} -> {ESCENARIOS[la.argmax().item()]}")
    print(f"{b!r:34s} -> {ESCENARIOS[lb.argmax().item()]}")
    print(f"   ¿los logits son idénticos? {torch.allclose(la, lb, atol=1e-5)}")
    print(f"   máxima diferencia entre los 18 logits: {(la - lb).abs().max():.2e}")
    print()

### Ejercicio 9 — El límite de esta arquitectura

**Objetivo:** Identificar con precisión qué información descarta el modelo, y por qué eso motiva la unidad siguiente.

**Enunciado:**

La celda de arriba pasa por el modelo tres pares de frases que usan **exactamente las mismas palabras en distinto orden**. Los *logits* no se parecen: son idénticos hasta la precisión de punto flotante.

Respondé:

1. **Explicá por qué tienen que ser idénticos.** Seguí la entrada a través del `forward` y señalá la operación exacta donde se pierde el orden.

2. **Construí un ejemplo propio**, del dominio de este corpus, donde la invariancia cause un error grave: dos órdenes con el mismo conjunto de palabras y significados claramente distintos. Después decidí: ¿alcanzaría con hacer el modelo más grande —más dimensiones, más capas— para resolverlo?

3. **Proponé, conceptualmente, qué habría que cambiar** en la arquitectura para que el orden importe. No hace falta que lo implementes ni que uses el nombre técnico: describí qué propiedad tendría que tener la operación que reemplace al promedio.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] La verificación del Ejercicio 3 da un valor cercano a `ln(18) = 2,890`.
- [ ] El entrenamiento del Ejercicio 5 llega a una accuracy de validación bastante por encima del 13,8% de la clase mayoritaria.
- [ ] Todos los gráficos tienen título, etiquetas en los ejes, leyenda y grilla.
- [ ] La matriz de confusión del Ejercicio 7 tiene los nombres de los 18 escenarios en los dos ejes y se lee.
- [ ] La brecha del Ejercicio 8 baja con la regularización, y la tabla comparativa está impresa.
- [ ] Los valores numéricos que imprimo son razonables (no hay infinitos, ni `NaN`, ni accuracies fuera de `[0, 1]`).
- [ ] Respondí las nueve preguntas de análisis (Ej. 1 a 9).
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Con esto cerrás el Laboratorio 1. Construiste, de punta a punta y sin cajas negras, un clasificador de texto en español:

- **El pipeline de datos**: `Dataset`, `DataLoader`, y por qué barajar el entrenamiento y no la evaluación.
- **El modelo**: `nn.Embedding`, promedio enmascarado y MLP, con el 95% de sus parámetros en la tabla de *embeddings*.
- **La verificación de cordura**: predecir `ln(C)` antes de entrenar, que es la manera más barata de detectar un bug de plomería.
- **El entrenamiento**: el paso a mano, el loop completo, y la comparación de tasas de aprendizaje y optimizadores con evidencia.
- **La evaluación**: matriz de confusión, precisión y *recall* por clase, y por qué la accuracy global esconde a las clases chicas.
- **El sobreajuste**: provocado a propósito, medido, y bajado con *weight decay* y *dropout*.

Y terminaste con un techo bien identificado: el modelo trata cada orden como una bolsa de palabras y no puede, ni en principio, distinguir *"reenviá el mensaje de mamá a papá"* de *"reenviá el mensaje de papá a mamá"*.

El **Laboratorio 2** ataca ese techo desde su base. Primero las representaciones: en lugar de dejar que la tabla de *embeddings* aprenda como subproducto de clasificar, vamos a entrenarla con un objetivo propio —word2vec, con muestreo negativo— y a comparar las dos geometrías que salen del mismo corpus. Después, la arquitectura: una red recurrente que procesa la orden palabra por palabra y para la cual el orden, por fin, significa algo.